# 강수–기온 분위회귀 · Solver 대안

Excel과 같은 102개 자료로 가중 절대오차 최소화

**할 일** · p=0.5·0.9·0.99의 기울기와 민감도 비교

위에서 아래로 실행. 코드를 바꾸기 전 결과를 예상하고, 실행 결과와 비교

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import linprog

url = 'https://raw.githubusercontent.com/steelpl/lecture/main/CE776-climate-change-carbon-water-cycle/week13/data/precipitation_temperature.csv'
data = pd.read_csv(url)
assert len(data)==102 and (data.precipitation_mm>0).all()
x = data.temperature_c.to_numpy()
y = np.log10(data.precipitation_mm.to_numpy())
X = np.column_stack([np.ones(len(x)), x])
data.head()

잔차 r = 관측 − 예측. r≥0의 손실은 p·r, r<0의 손실은 (1−p)·(−r). 잔차를 양·음 부분 u−v로 나누면 선형계획법으로 계산 가능

In [ ]:
def quantile_fit(p):
    n = len(y)
    # y - Xb = u - v, u>=0, v>=0; 절편·기울기에는 부호 제약 없음
    cost = np.r_[0., 0., np.full(n,p), np.full(n,1-p)]
    eq = np.column_stack([X, np.eye(n), -np.eye(n)])
    result = linprog(cost, A_eq=eq, b_eq=y,
                     bounds=[(None,None)]*2+[(0,None)]*(2*n), method="highs")
    if not result.success:
        raise RuntimeError(result.message)
    intercept, slope = result.x[:2]
    return {"p":p,"intercept":intercept,"slope":slope,
            "scaling_percent_per_K":100*(10**slope-1),"loss":result.fun}

fits = pd.DataFrame([quantile_fit(p) for p in [.5,.9,.99]])
fits

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7,4))
ax.scatter(x,y,s=16,alpha=.6)
xx = np.linspace(x.min(),x.max(),100)
for row in fits.itertuples():
    ax.plot(xx,row.intercept+row.slope*xx,label=f"q={row.p}")
ax.set(xlabel="Temperature (°C)",ylabel="log10 precipitation (mm)")
ax.legend(); plt.show()

p=.99가 p=.9보다 기울기가 큰지 확인하시오. 102개 관측으로 매우 높은 분위의 민감도를 추정할 때 불확실성이 커지는 이유를 설명하시오.

**비교 기준** · 약 4.8917·8.7056·6.7933%/K. 표본 연관성과 장기 기후변화 효과 구분. 표본의 관측소·기간 대표성은 별도 검토